# Video Action Recogni;on with UCF50 using LRCN

In [15]:
import os
import glob
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch import optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report,
    multilabel_confusion_matrix,
    roc_auc_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import GroupShuffleSplit

import sys
sys.path.append("/users/jmatthia/deep_learning/code/assignment_4/video_rec")

from utils import get_frames, store_frames
import shutil

Extracting frames with a unifor frequency of 16

In [11]:
video_dir = "/users/jmatthia/deep_learning/data/UCF50/UCF50"
frame_dir = "/users/jmatthia/deep_learning/data/UCF50_frames"

# choose every 16 frames, can be changed as needed 
n_frames = 16

for action_class in tqdm(sorted(os.listdir(video_dir))):
    for video_file in sorted(os.listdir(os.path.join(video_dir, action_class))):
        video_path = os.path.join(video_dir, action_class, video_file)
        store_path = os.path.join(frame_dir, action_class, os.path.splitext(video_file)[0])
        os.makedirs(store_path, exist_ok=True)
        frames, _ = get_frames(video_path, n_frames=n_frames)
        store_frames(frames, store_path)

  0%|                                                                                                                                                                                                                                      | 0/50 [00:00<?, ?it/s]

processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing

  2%|████▍                                                                                                                                                                                                                         | 1/50 [00:09<07:37,  9.35s/it]

processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing

  4%|████████▉                                                                                                                                                                                                                     | 2/50 [00:18<07:24,  9.26s/it]

...

processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌    | 49/50 [08:42<00:09,  9.35s/it]

processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [08:51<00:00, 10.64s/it]

processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing
processing


Creating a custom data split into train, validation and test (careful to avoid data leakage across action groups)

In [ ]:
def extract_group_id(vid_path):
    """Extract group ID from video folder name, e.g. g01 from v_BaseballPitch_g01_c01."""
    match = re.search(r'_g(\d+)_', os.path.basename(vid_path))
    return match.group(1) if match else None

# Build paths, labels, and group IDs
vid_paths, vid_labels, vid_groups = [], [], []
for action_class in sorted(os.listdir(frame_dir)):
    class_path = os.path.join(frame_dir, action_class)
    if not os.path.isdir(class_path):
        continue
    for vid_folder in sorted(os.listdir(class_path)):
        vid_path = os.path.join(class_path, vid_folder)
        group_id = extract_group_id(vid_folder)
        if group_id is None:
            continue
        vid_paths.append(vid_path)
        vid_labels.append(action_class)
        vid_groups.append(f"{action_class}_{group_id}")  # unique per class+group

vid_paths = np.array(vid_paths)
vid_labels = np.array(vid_labels)
vid_groups = np.array(vid_groups)

# First split off test set
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_val_idx, ts_idx = next(gss_test.split(vid_paths, vid_labels, vid_groups))

# Then split remaining into train/val
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss_val.split(vid_paths[tr_val_idx], vid_labels[tr_val_idx], vid_groups[tr_val_idx]))

tr_split  = list(zip(vid_paths[tr_val_idx][tr_idx],  vid_labels[tr_val_idx][tr_idx]))
val_split = list(zip(vid_paths[tr_val_idx][val_idx], vid_labels[tr_val_idx][val_idx]))
ts_split  = list(zip(vid_paths[ts_idx],              vid_labels[ts_idx]))

print(f"Train: {len(tr_split)} | Val: {len(val_split)} | Test: {len(ts_split)}")


Train: 4832 | Val: 853 | Test: 996


In [13]:
# Verify no group leakage
tr_groups  = set(vid_groups[tr_val_idx][tr_idx])
val_groups = set(vid_groups[tr_val_idx][val_idx])
ts_groups  = set(vid_groups[ts_idx])

assert len(tr_groups & val_groups) == 0, "Leakage: train/val share groups"
assert len(tr_groups & ts_groups)  == 0, "Leakage: train/test share groups"
assert len(val_groups & ts_groups) == 0, "Leakage: val/test share groups"
print("No group leakage detected.")

No group leakage detected.


In [27]:
split_dir = "/users/jmatthia/deep_learning/data/UCF50_splits"

for split_name, split_data in [("train", tr_split), ("val", val_split), ("test", ts_split)]:
    for vid_path, label in tqdm(split_data, desc=split_name):
        vid_path = str(vid_path)
        class_name = os.path.basename(os.path.dirname(vid_path))
        dst = os.path.join(split_dir, split_name, class_name, os.path.basename(vid_path))
        shutil.copytree(vid_path, dst, dirs_exist_ok=True)

print("Done.")

test: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 996/996 [01:17<00:00, 12.83it/s]

Done.


In [28]:
def get_groups(split_name):
    split_path = os.path.join(split_dir, split_name)
    groups = set()
    for cls in os.listdir(split_path):
        for vid in os.listdir(os.path.join(split_path, cls)):
            match = re.search(r'_g(\d+)_', vid)
            if match:
                groups.add(f"{cls}_{match.group(1)}")
    return groups

tr_g  = get_groups("train")
val_g = get_groups("val")
ts_g  = get_groups("test")

print(f"Train groups: {len(tr_g)}, Val groups: {len(val_g)}, Test groups: {len(ts_g)}")
print(f"Train/Val overlap: {len(tr_g & val_g)}")
print(f"Train/Test overlap: {len(tr_g & ts_g)}")
print(f"Val/Test overlap:   {len(val_g & ts_g)}")

Train groups: 912, Val groups: 162, Test groups: 190
Train/Val overlap: 0
Train/Test overlap: 0
Val/Test overlap:   0
